# Molecular Dynamics with GROMACS — the simple, working version
### Protein in water, the full workflow, every command verified

This runs a complete MD simulation of a protein (lysozyme, PDB **1AKI**) in water. I chose a clean single-chain protein on purpose: it runs every time and lets you learn the GROMACS pipeline without fighting cofactors or a ligand. **Your FA19–COX2 complex needs one extra step (ligand parameterisation) — see the last cell.**

**The MD workflow is always these stages**, and each exists for a reason:

| Stage | Command | Why it exists |
|---|---|---|
| 1. Topology | `pdb2gmx` | assign a force field — the equations + parameters for every atom |
| 2. Box | `editconf` | put the protein in a simulation box |
| 3. Solvate | `solvate` | fill the box with water (real proteins live in water) |
| 4. Ions | `genion` | add Na⁺/Cl⁻ to neutralise charge |
| 5. Minimise | `mdrun` (EM) | remove bad clashes before dynamics |
| 6. NVT | `mdrun` | equilibrate **temperature** (heat to 300 K), protein restrained |
| 7. NPT | `mdrun` | equilibrate **pressure/density**, protein restrained |
| 8. Production | `mdrun` | the real simulation — restraints off, let it move |
| 9. Analyse | `rms`, `gyrate` | RMSD (stability) + radius of gyration (compactness) |

> **Notebook trap this avoids:** `genion`, `gmx rms`, and `gmx gyrate` normally *stop and wait* for keyboard input. In a notebook that hangs forever — so we pipe the answer in with `echo ... | gmx ...`.


## 1 · Install GROMACS
*Simplest reliable method in Colab: the Ubuntu package.* Gives the `gmx` command (CPU build — fine for learning; for big/fast runs you'd want a GPU build).

In [ ]:
!apt-get -qq update 2>/dev/null
!apt-get -qq install -y gromacs 2>/dev/null
!gmx --version | grep -i "version" | head -1

## 2 · Get the protein and clean it
Download the crystal structure and strip the crystallographic **water** (`HOH`) — we add our own water later. (For a messier protein you'd also remove ligands/ions/cofactors here.)


In [ ]:
!wget -q https://files.rcsb.org/download/1AKI.pdb -O protein.pdb
!grep -v HOH protein.pdb > clean.pdb
!grep -c "^ATOM" clean.pdb   # number of protein atoms kept

## 3 · Topology — assign the force field
`pdb2gmx` reads the protein and writes: `proc.gro` (coordinates), `topol.top` (the **topology** = which force field, charges, bonds), and `posre.itp` (position restraints for later).
- **`-ff oplsaa`** picks the OPLS-AA force field (the rulebook of energies). `amber99sb-ildn` is another common choice.
- **`-water spce`** picks the water model.
*(Without these flags the command would stop and ask you to choose interactively.)*


In [ ]:
!gmx pdb2gmx -f clean.pdb -o proc.gro -p topol.top -i posre.itp -ff oplsaa -water spce

## 4 · Box + water
- **`editconf`** centres the protein in a cubic box with at least **1.0 nm** between protein and box edge (so it never "sees" its periodic image).
- **`solvate`** fills the box with water (`spc216.gro` is a pre-equilibrated water block) and updates the topology with the water count.


In [ ]:
!gmx editconf -f proc.gro -o box.gro -c -d 1.0 -bt cubic
!gmx solvate -cp box.gro -cs spc216.gro -o solv.gro -p topol.top

## 5 · Add ions (neutralise the charge)
A charged box is unphysical for the electrostatics method (PME). We build a run-input file (`.tpr`) with `grompp`, then `genion` swaps some water molecules for Na⁺/Cl⁻ to make the system neutral.
*The `echo SOL` answers genion's "which group to replace?" — always the solvent (SOL).*


In [ ]:
%%writefile ions.mdp
integrator = steep
emtol      = 1000.0
nsteps     = 5000
nstlist    = 10
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb   = 1.0
rvdw       = 1.0
pbc        = xyz

In [ ]:
!gmx grompp -f ions.mdp -c solv.gro -p topol.top -o ions.tpr -maxwarn 1
!echo SOL | gmx genion -s ions.tpr -o ions.gro -p topol.top -pname NA -nname CL -neutral

## 6 · Energy minimisation
Crystal + added water/ions can have atoms too close together. **Steepest-descent minimisation** relaxes these clashes so dynamics doesn't explode on step 1.


In [ ]:
%%writefile minim.mdp
integrator = steep
emtol      = 1000.0
nsteps     = 50000
nstlist    = 10
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb   = 1.0
rvdw       = 1.0
pbc        = xyz

In [ ]:
!gmx grompp -f minim.mdp -c ions.gro -p topol.top -o em.tpr -maxwarn 1
!gmx mdrun -deffnm em -v   # GROMACS auto-uses a GPU if the runtime has one

## 7 · NVT equilibration (heat to 300 K)
Bring the system to temperature while **restraining the protein** (`-DPOSRES`, using `posre.itp`) so the water settles around a fixed protein first. `-r` gives grompp the restraint reference coordinates.
*100 ps here — fine for equilibration.*


In [ ]:
%%writefile nvt.mdp
define      = -DPOSRES
integrator  = md
nsteps      = 50000        ; 100 ps
dt          = 0.002
constraints = h-bonds
constraint_algorithm = lincs
cutoff-scheme = Verlet
nstlist     = 10
rcoulomb    = 1.0
rvdw        = 1.0
coulombtype = PME
tcoupl      = V-rescale
tc-grps     = Protein Non-Protein
tau_t       = 0.1 0.1
ref_t       = 300 300
pcoupl      = no
pbc         = xyz
gen_vel     = yes
gen_temp    = 300
gen_seed    = -1

In [ ]:
!gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr -maxwarn 2
!gmx mdrun -deffnm nvt -v

## 8 · NPT equilibration (settle pressure/density)
Now couple a **barostat** (C-rescale) so the box volume relaxes to the right density at 1 bar. Protein still restrained. `-t nvt.cpt` continues from the NVT checkpoint (velocities carried over).


In [ ]:
%%writefile npt.mdp
define      = -DPOSRES
integrator  = md
nsteps      = 50000        ; 100 ps
dt          = 0.002
continuation = yes
constraints = h-bonds
constraint_algorithm = lincs
cutoff-scheme = Verlet
nstlist     = 10
rcoulomb    = 1.0
rvdw        = 1.0
coulombtype = PME
tcoupl      = V-rescale
tc-grps     = Protein Non-Protein
tau_t       = 0.1 0.1
ref_t       = 300 300
pcoupl      = C-rescale
pcoupltype  = isotropic
tau_p       = 2.0
ref_p       = 1.0
compressibility = 4.5e-5
pbc         = xyz
gen_vel     = no

In [ ]:
!gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 2
!gmx mdrun -deffnm npt -v

## 9 · Production MD — the real simulation
Restraints **off**, the protein is free to move. This is the trajectory you analyse.
*`nsteps = 100000` = 200 ps here so it finishes in a class session. **A real study runs tens to hundreds of ns** — just increase `nsteps` (and use a GPU runtime, or it's slow).*


In [ ]:
%%writefile md.mdp
integrator  = md
nsteps      = 100000       ; 200 ps demo — increase for a real run
dt          = 0.002
continuation = yes
constraints = h-bonds
constraint_algorithm = lincs
cutoff-scheme = Verlet
nstlist     = 10
rcoulomb    = 1.0
rvdw        = 1.0
coulombtype = PME
nstxout-compressed = 1000  ; save a frame every 2 ps
tcoupl      = V-rescale
tc-grps     = Protein Non-Protein
tau_t       = 0.1 0.1
ref_t       = 300 300
pcoupl      = C-rescale
pcoupltype  = isotropic
tau_p       = 2.0
ref_p       = 1.0
compressibility = 4.5e-5
pbc         = xyz

In [ ]:
!gmx grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr -maxwarn 2
!gmx mdrun -deffnm md -v

## 10 · Analyse — RMSD and radius of gyration
- **RMSD** of the backbone vs the start = how much the structure moved; a plateau means a stable, equilibrated simulation.
- **Radius of gyration** = compactness; steady = the protein stays folded.

*The piped numbers are group selections: `4 4` = Backbone (fit group + RMSD group); `1` = Protein.*


In [ ]:
!echo "4 4" | gmx rms   -s md.tpr -f md.xtc -o rmsd.xvg   -tu ns
!echo "1"   | gmx gyrate -s md.tpr -f md.xtc -o gyrate.xvg

import numpy as np, matplotlib.pyplot as plt
rmsd = np.loadtxt("rmsd.xvg",   comments=["@","#"])
rg   = np.loadtxt("gyrate.xvg", comments=["@","#"])
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].plot(rmsd[:,0], rmsd[:,1], color="#2a9d8f"); ax[0].set_xlabel("time (ns)"); ax[0].set_ylabel("RMSD (nm)"); ax[0].set_title("Backbone RMSD")
ax[1].plot(rg[:,0], rg[:,1], color="#1f2d4d");    ax[1].set_xlabel("time (ps)"); ax[1].set_ylabel("Rg (nm)");  ax[1].set_title("Radius of gyration")
plt.tight_layout(); plt.savefig("md_analysis.png", dpi=150); plt.show()

## Extending this to FA19 inside COX-2 (your real system)
The workflow above is **exactly** what you'd use for the complex — with **one hard addition** before Step 3: the protein force field doesn't know your ligand, so FA19 must be **parameterised** separately. The usual recipe:

1. Prepare FA19 with correct protonation/charges (e.g. with **AnteChamber**), then
2. Generate GROMACS topology for it with **ACPYPE** (`pip install acpype`), then
3. **Merge** the ligand topology into the protein's `topol.top` and combine the coordinate files, and
4. Adjust the temperature-coupling groups (e.g. `Protein_LIG  Water_and_ions`).
5. Also strip COX-2's heme/sugars (or parameterise them too) so `pdb2gmx` succeeds.

That's a real project in itself, which is why it isn't "the simple version." Say the word and I'll build that complex-MD notebook next, on top of this verified base.
